In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shuvoalok/raf-db-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'raf-db-dataset' dataset.
Path to dataset files: /kaggle/input/raf-db-dataset


In [ ]:
import os

# Gather unique directory paths containing images
unique_dirs = set()
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            # Get the path relative to the directory containing 'kdef-database'
            base_dir = os.path.dirname(os.path.dirname(os.path.dirname(path)))
            relative_dir = os.path.relpath(root, base_dir)
            unique_dirs.add(relative_dir)

# Print the paths (without filenames)
for d in sorted(list(unique_dirs)):
    print(f"{d}/")

kaggle/input/raf-db-dataset/DATASET/test/1/
kaggle/input/raf-db-dataset/DATASET/test/2/
kaggle/input/raf-db-dataset/DATASET/test/3/
kaggle/input/raf-db-dataset/DATASET/test/4/
kaggle/input/raf-db-dataset/DATASET/test/5/
kaggle/input/raf-db-dataset/DATASET/test/6/
kaggle/input/raf-db-dataset/DATASET/test/7/
kaggle/input/raf-db-dataset/DATASET/train/1/
kaggle/input/raf-db-dataset/DATASET/train/2/
kaggle/input/raf-db-dataset/DATASET/train/3/
kaggle/input/raf-db-dataset/DATASET/train/4/
kaggle/input/raf-db-dataset/DATASET/train/5/
kaggle/input/raf-db-dataset/DATASET/train/6/
kaggle/input/raf-db-dataset/DATASET/train/7/


In [ ]:
import os
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

# 1. Gather all image files from the dataset path
image_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            image_files.append(os.path.join(root, file))

print(f"Found {len(image_files)} images in the dataset.")

Found 15339 images in the dataset.


In [ ]:
# 2. Define the preprocessing pipeline
# Standard ImageNet normalization values are often used when resizing to 224x224
preprocess_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),        # Resize to 224x224
    transforms.Lambda(lambda x: x.convert("RGB")), # Convert to RGB (3 channels)
    transforms.ToTensor(),                # Convert to Tensor (0-1 range)
    transforms.Normalize(                 # Normalize
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Apply to a sample image
if image_files:
    sample_image_path = image_files[0]
    original_image = Image.open(sample_image_path)

    # Apply preprocessing
    processed_tensor = preprocess_pipeline(original_image)

    print(f"Original Size: {original_image.size}")
    print(f"Processed Tensor Shape: {processed_tensor.shape}")
    print(f"Tensor Stats - Min: {processed_tensor.min():.3f}, Max: {processed_tensor.max():.3f}, Mean: {processed_tensor.mean():.3f}")
else:
    print("No images found to process.")

Original Size: (100, 100)
Processed Tensor Shape: torch.Size([3, 224, 224])
Tensor Stats - Min: -2.001, Max: 2.112, Mean: 0.115


In [ ]:
import os
import torch
from PIL import Image
from tqdm import tqdm

# Define output directory in Drive
output_dir = '/content/drive/MyDrive/processed_raf_db_pt'

print(f"Saving processed tensors to: {output_dir}")

# Map numeric labels to class names
class_mapping = {
    '1': 'Surprise',
    '2': 'Fear',
    '3': 'Disgust',
    '4': 'Happy',
    '5': 'Sad',
    '6': 'Angry',
    '7': 'Neutral'
}

# Process and save all images
successful_saves = 0

for img_path in tqdm(image_files, desc="Processing Images"):
    try:
        # Open image
        img = Image.open(img_path)

        # Apply full preprocessing pipeline (returns a torch.Tensor)
        processed_tensor = preprocess_pipeline(img)

        # Extract train/test split from the parent of the parent directory
        split_name = os.path.basename(os.path.dirname(os.path.dirname(img_path)))

        # Extract class number (the immediate parent directory of the image)
        class_num = os.path.basename(os.path.dirname(img_path))

        # Get the class name, default to the number if not found in mapping
        class_name = class_mapping.get(class_num, class_num)

        # Get original filename
        filename = os.path.basename(img_path)

        # Strip the first prefix (e.g., 'train' or 'test') and prepend 'raf_db_'
        parts = filename.split('_', 1)
        if len(parts) > 1:
            new_filename = f"raf_db_{parts[1]}"
        else:
            new_filename = f"raf_db_{filename}"

        filename_pt = os.path.splitext(new_filename)[0] + '.pt'

        # Construct full save path including train/test split
        save_path = os.path.join(output_dir, split_name, class_name, filename_pt)

        # Ensure the subdirectory exists
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # Save as a PyTorch tensor file
        torch.save(processed_tensor, save_path)
        successful_saves += 1
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

print(f"Successfully saved {successful_saves} tensors to Google Drive, categorized by split and class name.")


Saving processed tensors to: /content/drive/MyDrive/processed_raf_db_pt


Processing Images: 100%|██████████| 15339/15339 [08:56<00:00, 28.57it/s]

Successfully saved 15339 tensors to Google Drive, categorized by split and class name.


In [ ]:
import os

dir_path = '/content/drive/MyDrive/processed_raf_db_pt'

if os.path.exists(dir_path):
    total_files = sum(len(files) for root, dirs, files in os.walk(dir_path))
    print(f"Total files in '{dir_path}': {total_files}")
else:
    print(f"Directory '{dir_path}' does not exist.")

Total files in '/content/drive/MyDrive/processed_raf_db_pt': 15339


In [ ]:
import os
import pandas as pd
from collections import Counter

# Path to the directory where we saved the splits (Fixed the typo from '-' to '_')
dir_path = '/content/drive/MyDrive/processed_raf_db_pt'

train_counts = Counter()
test_counts = Counter()

# Count images by split and class directly from the folders
if os.path.exists(dir_path):
    for split_name in ['train', 'test']:
        split_path = os.path.join(dir_path, split_name)
        if os.path.exists(split_path):
            for class_name in os.listdir(split_path):
                class_dir = os.path.join(split_path, class_name)
                if os.path.isdir(class_dir):
                    num_files = len([f for f in os.listdir(class_dir) if f.endswith('.pt')])
                    if split_name == 'train':
                        train_counts[class_name] += num_files
                    elif split_name == 'test':
                        test_counts[class_name] += num_files

# Get all unique classes
all_classes = sorted(list(set(train_counts.keys()).union(set(test_counts.keys()))))

# Create a DataFrame for visualization
df_counts = pd.DataFrame({
    'Class': all_classes,
    'Train Count': [train_counts.get(c, 0) for c in all_classes],
    'Test Count': [test_counts.get(c, 0) for c in all_classes]
})

# Add Total column
df_counts['Total'] = df_counts['Train Count'] + df_counts['Test Count']

# Add a Total row at the bottom
total_row = pd.DataFrame({
    'Class': ['All Classes'],
    'Train Count': [df_counts['Train Count'].sum()],
    'Test Count': [df_counts['Test Count'].sum()],
    'Total': [df_counts['Total'].sum()]
})

df_counts = pd.concat([df_counts, total_row], ignore_index=True)

# Display the results
display(df_counts)


,Class,Train Count,Test Count,Total
0,Angry,705,162,867
1,Disgust,717,160,877
2,Fear,281,74,355
3,Happy,4772,1185,5957
4,Neutral,2524,680,3204
5,Sad,1982,478,2460
6,Surprise,1290,329,1619
7,All Classes,12271,3068,15339


In [ ]:
from google.colab import runtime
runtime.unassign()